# Model training v2 (leak-free)
Changes vs `model_train.ipynb`:
- dropped `Brand_avgPrice` / `Model_avgPrice` (they were computed from `AskPrice` over the whole dataset = target leakage)
- target is `log(AskPrice)`; predictions are converted back with `exp`
- `Brand` / `model` are given to CatBoost as categorical columns (it encodes them internally without leaking)
- added `km_per_year`, `log_km`

In [ ]:
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error
from catboost import CatBoostRegressor

df = pd.read_csv(Path('../data')/'clean.csv')
df = df.drop(columns=['Brand_avgPrice', 'Model_avgPrice'])   # remove the leaky columns

df['km_per_year'] = df['kmDriven'] / df['Age'].clip(lower=1)
df['log_km'] = np.log1p(df['kmDriven'])
df['Brand_model'] = df['Brand'] + ' ' + df['model']

cat_cols = ['Brand', 'model', 'Brand_model', 'FuelType']
X = df.drop(columns=['AskPrice'])
y = np.log(df['AskPrice'])          # log of AskPrice is the TARGET, not a feature
X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def make_model():
    return CatBoostRegressor(iterations=500, learning_rate=0.1, depth=6,
                             cat_features=cat_cols, random_seed=42, verbose=0)

model = make_model().fit(X_train, y_train)

In [ ]:
def report(y_true_log, y_pred_log):
    y_true, y_pred = np.exp(y_true_log), np.exp(y_pred_log)   # back to rupees
    print(f"R2 (rupees)        : {r2_score(y_true, y_pred):.4f}")
    print(f"R2 (log scale)     : {r2_score(y_true_log, y_pred_log):.4f}")
    print(f"MAE                : {mean_absolute_error(y_true, y_pred):,.0f}")
    print(f"MAPE               : {mean_absolute_percentage_error(y_true, y_pred):.3f}")
    print(f"median % error     : {np.median(np.abs(y_pred - y_true) / y_true):.3f}")

report(y_test, model.predict(X_test))

In [ ]:
# 5-fold CV (encoding happens inside each fold, so no leakage)
scores = []
for tr, va in KFold(5, shuffle=True, random_state=42).split(X):
    m = make_model().fit(X.iloc[tr], y.iloc[tr])
    scores.append(r2_score(np.exp(y.iloc[va]), np.exp(m.predict(X.iloc[va]))))
print([round(s, 3) for s in scores], 'mean R2 =', round(np.mean(scores), 4))

In [ ]:
pd.Series(model.get_feature_importance(), index=X.columns).sort_values().plot.barh(figsize=(6, 4))